In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'pharma_cold_chain_raw.csv')

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nFirst 5 rows:")
df.head()

Shape: (515, 15)

Columns: ['Shipment_ID', 'Product', 'Route', 'Vendor', 'Dispatch_Timestamp', 'Delivery_Timestamp', 'Transit_Hours', 'Avg_Temperature_C', 'Max_Temperature_C', 'Min_Temperature_C', 'Breach_Hours', 'Batch_Value_Lakhs', 'Destination', 'Compliance_Status', 'Num_Temp_Readings']

First 5 rows:


,Shipment_ID,Product,Route,Vendor,Dispatch_Timestamp,Delivery_Timestamp,Transit_Hours,Avg_Temperature_C,Max_Temperature_C,Min_Temperature_C,Breach_Hours,Batch_Value_Lakhs,Destination,Compliance_Status,Num_Temp_Readings
0,SHP-1381,Adalimumab,Mumbai-Hyderabad,ChillChain,16/01/2024,18/01/2024,51,5.07,7.44,2.60,0,75.63,Chennai,Compliant,51
1,SHP-1200,Insulin,Mumbai-Hyderabad,MediCargo Exp.,30/07/2024,02/08/2024,70,5.21,7.72,2.66,0,63.73 L,Kolkata,Compliant,70
2,SHP-1314,Adalimumab,Mumbai-Delhi,ChillChain Pvt Ltd,2024-07-31 19:00:00,2024-08-04 08:00:00,85,5.10,7.73,2.51,0,59.68,Hyderabad,Compliant,85
3,SHP-1384,MMR Vaccine,Mumbai-Delhi,pharmafreight inc,2024-07-05 11:00:00,2024-07-05 09:00:00,82,4.95,7.49,2.52,0,12.72,Hyderabad,Compliant,82
4,SHP-1039,Trastuzumab,Delhi-Jaipur,Medi Cargo Express,2024-06-10,2024-06-14,74,6.25,11.59,2.53,15,62.28,Kolkata,Non-Compliant,74


In [2]:
# Step 2 — Understand the mess before cleaning
print("=== DATA TYPES ===")
print(df.dtypes)

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DUPLICATE ROWS ===")
print("Total duplicates:", df.duplicated().sum())

print("\n=== SAMPLE OF MESSY COLUMNS ===")
print("\nVendor (first 20 unique values):")
print(df['Vendor'].unique()[:20])

print("\nBatch_Value_Lakhs sample (showing dirty values):")
print(df['Batch_Value_Lakhs'].head(20).tolist())

print("\nTimestamp format mess:")
print(df['Dispatch_Timestamp'].head(10).tolist())

=== DATA TYPES ===
Shipment_ID            object
Product                object
Route                  object
Vendor                 object
Dispatch_Timestamp     object
Delivery_Timestamp     object
Transit_Hours           int64
Avg_Temperature_C     float64
Max_Temperature_C     float64
Min_Temperature_C     float64
Breach_Hours            int64
Batch_Value_Lakhs      object
Destination            object
Compliance_Status      object
Num_Temp_Readings       int64
dtype: object

=== MISSING VALUES ===
Shipment_ID            0
Product                0
Route                  0
Vendor                23
Dispatch_Timestamp     0
Delivery_Timestamp     0
Transit_Hours          0
Avg_Temperature_C     27
Max_Temperature_C     21
Min_Temperature_C      0
Breach_Hours           0
Batch_Value_Lakhs      0
Destination           22
Compliance_Status     22
Num_Temp_Readings      0
dtype: int64

=== DUPLICATE ROWS ===
Total duplicates: 14

=== SAMPLE OF MESSY COLUMNS ===

Vendor (first 20 unique va

In [3]:
# Step 3 — DATA CLEANING

df_clean = df.copy()  # always work on a copy, never destroy raw data

# ----- FIX 1: Remove duplicates -----
df_clean = df_clean.drop_duplicates()
print(f"After removing duplicates: {len(df_clean)} rows (removed {len(df) - len(df_clean)})")

# ----- FIX 2: Clean Batch_Value_Lakhs -----
# Remove ' L' suffix and convert to float
df_clean['Batch_Value_Lakhs'] = df_clean['Batch_Value_Lakhs'].str.replace(' L', '', regex=False)
df_clean['Batch_Value_Lakhs'] = pd.to_numeric(df_clean['Batch_Value_Lakhs'], errors='coerce')
print(f"\nBatch_Value_Lakhs dtype now: {df_clean['Batch_Value_Lakhs'].dtype}")

# ----- FIX 3: Standardize Vendor names -----
# Map all variants to one clean name
vendor_map = {
    'ColdEx Logistics'  : 'ColdEx Logistics',
    'coldex logistics'  : 'ColdEx Logistics',
    'ColdEx Logi.'      : 'ColdEx Logistics',
    'COLDEX LOGISTICS'  : 'ColdEx Logistics',

    'PharmaFreight Inc' : 'PharmaFreight Inc',
    'Pharma Freight Inc': 'PharmaFreight Inc',
    'pharmafreight inc' : 'PharmaFreight Inc',
    'PharmaFreight'     : 'PharmaFreight Inc',

    'BioShip Solutions' : 'BioShip Solutions',
    'Bioship Solutions' : 'BioShip Solutions',
    'BIOSHIP SOLUTIONS' : 'BioShip Solutions',
    'Bio Ship Solutions': 'BioShip Solutions',

    'MediCargo Express' : 'MediCargo Express',
    'Medi Cargo Express': 'MediCargo Express',
    'medicargo express' : 'MediCargo Express',
    'MediCargo Exp.'    : 'MediCargo Express',

    'ChillChain Pvt Ltd': 'ChillChain Pvt Ltd',
    'Chill Chain Pvt Ltd':'ChillChain Pvt Ltd',
    'chillchain pvt ltd': 'ChillChain Pvt Ltd',
    'ChillChain'        : 'ChillChain Pvt Ltd',
}
df_clean['Vendor'] = df_clean['Vendor'].map(vendor_map)
print(f"\nUnique vendors after cleaning: {df_clean['Vendor'].nunique()}")
print(df_clean['Vendor'].value_counts())

# ----- FIX 4: Fix timestamps -----
df_clean['Dispatch_Timestamp'] = pd.to_datetime(df_clean['Dispatch_Timestamp'], dayfirst=True, errors='coerce')
df_clean['Delivery_Timestamp'] = pd.to_datetime(df_clean['Delivery_Timestamp'], dayfirst=True, errors='coerce')
print(f"\nDispatch_Timestamp dtype now: {df_clean['Dispatch_Timestamp'].dtype}")

# ----- FIX 5: Remove impossible timestamps -----
# Delivery cannot be before or equal to dispatch
impossible = df_clean['Delivery_Timestamp'] <= df_clean['Dispatch_Timestamp']
print(f"\nImpossible timestamps found: {impossible.sum()}")
df_clean = df_clean[~impossible]
print(f"Rows after removing impossible timestamps: {len(df_clean)}")

# ----- FIX 6: Handle missing values -----
# Temperature - fill with median (robust to outliers)
df_clean['Avg_Temperature_C'] = df_clean['Avg_Temperature_C'].fillna(df_clean['Avg_Temperature_C'].median())
df_clean['Max_Temperature_C'] = df_clean['Max_Temperature_C'].fillna(df_clean['Max_Temperature_C'].median())

# Vendor and Destination - fill with 'Unknown'
df_clean['Vendor']      = df_clean['Vendor'].fillna('Unknown')
df_clean['Destination'] = df_clean['Destination'].fillna('Unknown')

# Compliance_Status - derive from Breach_Hours where missing
def derive_compliance(row):
    if pd.isna(row['Compliance_Status']):
        if row['Breach_Hours'] == 0:
            return 'Compliant'
        elif row['Breach_Hours'] <= 4:
            return 'At-Risk'
        else:
            return 'Non-Compliant'
    return row['Compliance_Status']

df_clean['Compliance_Status'] = df_clean.apply(derive_compliance, axis=1)

print(f"\nMissing values after cleaning:")
print(df_clean.isnull().sum())

After removing duplicates: 501 rows (removed 14)

Batch_Value_Lakhs dtype now: float64

Unique vendors after cleaning: 5
Vendor
PharmaFreight Inc     108
ColdEx Logistics      107
BioShip Solutions      95
ChillChain Pvt Ltd     84
MediCargo Express      84
Name: count, dtype: int64

Dispatch_Timestamp dtype now: datetime64[ns]

Impossible timestamps found: 7
Rows after removing impossible timestamps: 494

Missing values after cleaning:
Shipment_ID             0
Product                 0
Route                   0
Vendor                  0
Dispatch_Timestamp    379
Delivery_Timestamp    379
Transit_Hours           0
Avg_Temperature_C       0
Max_Temperature_C       0
Min_Temperature_C       0
Breach_Hours            0
Batch_Value_Lakhs       0
Destination             0
Compliance_Status       0
Num_Temp_Readings       0
dtype: int64


In [4]:
# Fix timestamp parsing — handle mixed formats explicitly

def parse_mixed_dates(date_str):
    if pd.isna(date_str):
        return pd.NaT
    
    formats = [
        '%d-%m-%Y %H:%M',      # 20-05-2024 15:00
        '%d/%m/%Y',            # 25/10/2024
        '%Y-%m-%d %H:%M:%S',   # 2024-06-08 09:00:00
        '%Y-%m-%d %H:%M',      # 2024-06-08 09:00
        '%Y-%m-%d',            # 2024-05-21
        '%d-%m-%Y',            # 20-05-2024
    ]
    
    for fmt in formats:
        try:
            return pd.to_datetime(date_str, format=fmt)
        except:
            continue
    return pd.NaT  # if nothing works, return Not a Time

# Re-parse on the original df_clean (before we lost timestamps)
df_clean = df.copy()

# Redo all cleaning steps cleanly
df_clean = df_clean.drop_duplicates()
df_clean['Batch_Value_Lakhs'] = df_clean['Batch_Value_Lakhs'].str.replace(' L', '', regex=False)
df_clean['Batch_Value_Lakhs'] = pd.to_numeric(df_clean['Batch_Value_Lakhs'], errors='coerce')
df_clean['Vendor'] = df_clean['Vendor'].map(vendor_map)

# Now parse timestamps properly
df_clean['Dispatch_Timestamp'] = df_clean['Dispatch_Timestamp'].apply(parse_mixed_dates)
df_clean['Delivery_Timestamp'] = df_clean['Delivery_Timestamp'].apply(parse_mixed_dates)

print(f"Missing Dispatch after proper parsing: {df_clean['Dispatch_Timestamp'].isna().sum()}")
print(f"Missing Delivery after proper parsing: {df_clean['Delivery_Timestamp'].isna().sum()}")

# Remove impossible timestamps
impossible = df_clean['Delivery_Timestamp'] <= df_clean['Dispatch_Timestamp']
print(f"Impossible timestamps: {impossible.sum()}")
df_clean = df_clean[~impossible]

# Handle missing values
df_clean['Avg_Temperature_C'] = df_clean['Avg_Temperature_C'].fillna(df_clean['Avg_Temperature_C'].median())
df_clean['Max_Temperature_C'] = df_clean['Max_Temperature_C'].fillna(df_clean['Max_Temperature_C'].median())
df_clean['Vendor']      = df_clean['Vendor'].fillna('Unknown')
df_clean['Destination'] = df_clean['Destination'].fillna('Unknown')
df_clean['Compliance_Status'] = df_clean.apply(derive_compliance, axis=1)

print(f"\nFinal clean dataset: {len(df_clean)} rows")
print(f"\nMissing values:")
print(df_clean.isnull().sum())
print(f"\nTimestamp sample:")
print(df_clean['Dispatch_Timestamp'].head(5).tolist())

Missing Dispatch after proper parsing: 0
Missing Delivery after proper parsing: 0
Impossible timestamps: 16

Final clean dataset: 485 rows

Missing values:
Shipment_ID           0
Product               0
Route                 0
Vendor                0
Dispatch_Timestamp    0
Delivery_Timestamp    0
Transit_Hours         0
Avg_Temperature_C     0
Max_Temperature_C     0
Min_Temperature_C     0
Breach_Hours          0
Batch_Value_Lakhs     0
Destination           0
Compliance_Status     0
Num_Temp_Readings     0
dtype: int64

Timestamp sample:
[Timestamp('2024-01-16 00:00:00'), Timestamp('2024-07-30 00:00:00'), Timestamp('2024-07-31 19:00:00'), Timestamp('2024-06-10 00:00:00'), Timestamp('2024-03-22 08:00:00')]


In [5]:
# Step 4 — Feature Engineering
# Extract month and season from dispatch timestamp

df_clean['Month'] = df_clean['Dispatch_Timestamp'].dt.month
df_clean['Month_Name'] = df_clean['Dispatch_Timestamp'].dt.strftime('%B')

def get_season(month):
    if month in [3, 4, 5]:
        return 'Summer'
    elif month in [6, 7, 8, 9]:
        return 'Monsoon'
    elif month in [10, 11]:
        return 'Post-Monsoon'
    else:
        return 'Winter'

df_clean['Season'] = df_clean['Month'].apply(get_season)

# Financial risk column
# Batch value is at risk only if Non-Compliant
df_clean['Value_At_Risk'] = df_clean.apply(
    lambda row: row['Batch_Value_Lakhs'] if row['Compliance_Status'] == 'Non-Compliant' else 0,
    axis=1
)

print("New columns added:")
print(df_clean[['Dispatch_Timestamp', 'Month_Name', 'Season', 'Batch_Value_Lakhs', 'Value_At_Risk']].head(10))

print(f"\nSeason distribution:")
print(df_clean['Season'].value_counts())

print(f"\nTotal financial value at risk: ₹{df_clean['Value_At_Risk'].sum():.2f} Lakhs")

New columns added:
    Dispatch_Timestamp Month_Name   Season  Batch_Value_Lakhs  Value_At_Risk
0  2024-01-16 00:00:00    January   Winter              75.63           0.00
1  2024-07-30 00:00:00       July  Monsoon              63.73           0.00
2  2024-07-31 19:00:00       July  Monsoon              59.68           0.00
4  2024-06-10 00:00:00       June  Monsoon              62.28          62.28
6  2024-03-22 08:00:00      March   Summer              10.57           0.00
7  2024-03-31 07:00:00      March   Summer              27.05           0.00
8  2024-01-18 18:00:00    January   Winter              71.66           0.00
9  2024-12-04 13:00:00   December   Winter              66.02           0.00
10 2024-08-27 00:00:00     August  Monsoon              65.62           0.00
11 2024-07-06 16:00:00       July  Monsoon              67.72           0.00

Season distribution:
Season
Monsoon         184
Winter          128
Summer          104
Post-Monsoon     69
Name: count, dtype: int64

In [6]:
# Save clean dataset
df_clean.to_csv(r'pharma_cold_chain_clean.csv', index=False)
print("Clean dataset saved successfully!")
print(f"Final shape: {df_clean.shape}")

Clean dataset saved successfully!
Final shape: (485, 19)


In [7]:
# Step 5 — VENDOR ANALYSIS
# Key question: Which vendors are responsible for the most failures?

vendor_analysis = df_clean.groupby('Vendor').agg(
    Total_Shipments    = ('Shipment_ID', 'count'),
    Non_Compliant      = ('Compliance_Status', lambda x: (x == 'Non-Compliant').sum()),
    At_Risk            = ('Compliance_Status', lambda x: (x == 'At-Risk').sum()),
    Compliant          = ('Compliance_Status', lambda x: (x == 'Compliant').sum()),
    Avg_Temp           = ('Avg_Temperature_C', 'mean'),
    Max_Temp_Recorded  = ('Max_Temperature_C', 'max'),
    Avg_Breach_Hours   = ('Breach_Hours', 'mean'),
    Total_Value_At_Risk= ('Value_At_Risk', 'sum')
).reset_index()

# Failure rate = Non-Compliant / Total
vendor_analysis['Failure_Rate_%'] = (
    vendor_analysis['Non_Compliant'] / vendor_analysis['Total_Shipments'] * 100
).round(2)

# Sort by failure rate
vendor_analysis = vendor_analysis.sort_values('Failure_Rate_%', ascending=False)

print("=== VENDOR RISK ANALYSIS ===\n")
print(vendor_analysis[['Vendor', 'Total_Shipments', 'Non_Compliant', 
                         'Failure_Rate_%', 'Avg_Breach_Hours', 
                         'Total_Value_At_Risk']].to_string(index=False))

=== VENDOR RISK ANALYSIS ===

            Vendor  Total_Shipments  Non_Compliant  Failure_Rate_%  Avg_Breach_Hours  Total_Value_At_Risk
  ColdEx Logistics              106             25           23.58          2.594340              1082.38
 MediCargo Express               83             12           14.46          1.963855               646.31
 BioShip Solutions               90             10           11.11          1.088889               358.66
ChillChain Pvt Ltd               79              4            5.06          0.632911               181.48
 PharmaFreight Inc              104              5            4.81          0.682692               263.20
           Unknown               23              1            4.35          0.913043                34.19


In [8]:
# Step 6 — ROUTE ANALYSIS
# Key question: Which routes are most dangerous?

route_analysis = df_clean.groupby('Route').agg(
    Total_Shipments    = ('Shipment_ID', 'count'),
    Non_Compliant      = ('Compliance_Status', lambda x: (x == 'Non-Compliant').sum()),
    At_Risk            = ('Compliance_Status', lambda x: (x == 'At-Risk').sum()),
    Avg_Breach_Hours   = ('Breach_Hours', 'mean'),
    Max_Temp_Recorded  = ('Max_Temperature_C', 'max'),
    Total_Value_At_Risk= ('Value_At_Risk', 'sum')
).reset_index()

route_analysis['Failure_Rate_%'] = (
    route_analysis['Non_Compliant'] / route_analysis['Total_Shipments'] * 100
).round(2)

route_analysis = route_analysis.sort_values('Failure_Rate_%', ascending=False)

print("=== ROUTE RISK ANALYSIS ===\n")
print(route_analysis[['Route', 'Total_Shipments', 'Non_Compliant',
                        'Failure_Rate_%', 'Avg_Breach_Hours',
                        'Total_Value_At_Risk']].to_string(index=False))

=== ROUTE RISK ANALYSIS ===

            Route  Total_Shipments  Non_Compliant  Failure_Rate_%  Avg_Breach_Hours  Total_Value_At_Risk
     Mumbai-Delhi               62             12           19.35          1.903226               559.81
     Delhi-Jaipur               59             10           16.95          1.847458               527.67
Chennai-Bangalore               65              7           10.77          1.492308               263.57
Bangalore-Chennai               56              6           10.71          1.428571               232.04
 Hyderabad-Mumbai               63              6            9.52          1.142857               257.17
 Kolkata-Guwahati               53              5            9.43          1.094340               131.68
 Mumbai-Hyderabad               57              5            8.77          1.122807               352.02
    Delhi-Kolkata               70              6            8.57          1.142857               242.26


In [10]:
worst = vendor_analysis.head(2)
reliable = vendor_analysis.tail(3)

print(f"\n--- WORST VENDORS ---")
for i, (_, row) in enumerate(worst.iterrows(), 1):
    print(f"{i}. {row['Vendor']:<25} → {row['Failure_Rate_%']}% failure rate")

print(f"\n--- RELIABLE VENDORS ---")
for i, (_, row) in enumerate(reliable.iterrows(), 1):
    print(f"{i}. {row['Vendor']:<25} → {row['Failure_Rate_%']}% failure rate")


--- WORST VENDORS ---
1. ColdEx Logistics          → 23.58% failure rate
2. MediCargo Express         → 14.46% failure rate

--- RELIABLE VENDORS ---
1. ChillChain Pvt Ltd        → 5.06% failure rate
2. PharmaFreight Inc         → 4.81% failure rate
3. Unknown                   → 4.35% failure rate


In [14]:
# Save summary to a text file
summary = f"""
==================================================
PYTHON ANALYSIS COMPLETE
==================================================

Original dataset:     515 rows
After cleaning:       {len(df_clean)} rows
Removed:              {515 - len(df_clean)} rows (duplicates + impossible timestamps)

--- KEY FINDINGS ---
Total shipments analysed:     {len(df_clean)}
Non-Compliant shipments:      {(df_clean['Compliance_Status'] == 'Non-Compliant').sum()}
Overall failure rate:         {(df_clean['Compliance_Status'] == 'Non-Compliant').sum() / len(df_clean) * 100:.2f}%
Total value at risk:          ₹{df_clean['Value_At_Risk'].sum():.2f} Lakhs

--- VENDOR FAILURE RATES ---
"""

for _, row in vendor_analysis.iterrows():
    summary += f"{row['Vendor']:<25} → {row['Failure_Rate_%']}%\n"

# Save to file
with open(r'C:\Users\kesha\OneDrive\Desktop\cold_chain_project\python_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary)

print("Summary saved → python_summary.txt")

Summary saved → python_summary.txt


In [15]:
# Verify failure rates in clean data
verification = df_clean.groupby('Vendor').agg(
    Total_Shipments = ('Shipment_ID', 'count'),
    Non_Compliant   = ('Compliance_Status', lambda x: (x == 'Non-Compliant').sum()),
    At_Risk         = ('Compliance_Status', lambda x: (x == 'At-Risk').sum()),
    Compliant       = ('Compliance_Status', lambda x: (x == 'Compliant').sum()),
).reset_index()

verification['Failure_Rate_%']  = (verification['Non_Compliant'] / verification['Total_Shipments'] * 100).round(2)
verification['AtRisk_Rate_%']   = (verification['At_Risk']       / verification['Total_Shipments'] * 100).round(2)
verification['Compliant_Rate_%']= (verification['Compliant']     / verification['Total_Shipments'] * 100).round(2)

verification = verification.sort_values('Failure_Rate_%', ascending=False)

print("=== FAILURE RATE VERIFICATION ===\n")
print(verification[['Vendor', 'Total_Shipments', 'Non_Compliant', 
                     'At_Risk', 'Compliant', 'Failure_Rate_%',
                     'AtRisk_Rate_%', 'Compliant_Rate_%']].to_string(index=False))

print(f"\n=== OVERALL ===")
total = len(df_clean)
nc    = (df_clean['Compliance_Status'] == 'Non-Compliant').sum()
ar    = (df_clean['Compliance_Status'] == 'At-Risk').sum()
co    = (df_clean['Compliance_Status'] == 'Compliant').sum()
print(f"Total Shipments:   {total}")
print(f"Non-Compliant:     {nc} ({nc/total*100:.2f}%)")
print(f"At-Risk:           {ar} ({ar/total*100:.2f}%)")
print(f"Compliant:         {co} ({co/total*100:.2f}%)")

=== FAILURE RATE VERIFICATION ===

            Vendor  Total_Shipments  Non_Compliant  At_Risk  Compliant  Failure_Rate_%  AtRisk_Rate_%  Compliant_Rate_%
  ColdEx Logistics              106             25       10         71           23.58           9.43             66.98
 MediCargo Express               83             12       11         60           14.46          13.25             72.29
 BioShip Solutions               90             10        4         76           11.11           4.44             84.44
ChillChain Pvt Ltd               79              4        0         75            5.06           0.00             94.94
 PharmaFreight Inc              104              5        3         96            4.81           2.88             92.31
           Unknown               23              1        2         20            4.35           8.70             86.96

=== OVERALL ===
Total Shipments:   485
Non-Compliant:     57 (11.75%)
At-Risk:           30 (6.19%)
Compliant:         398 (

In [16]:
df_clean.to_csv(r'pharma_cold_chain_clean.csv', index=False)
print("Saved successfully!")

Saved successfully!


In [12]:
with open(r'C:\Users\kesha\OneDrive\Desktop\cold_chain_project\python_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary)